## 問題
- 10台の製造装置を使い4種の製品を合計100個作りたい。
- 製造する品種を変更するには、部品の交換するための段取り時間が必要。
- 全数の生産が完了するまでの時間を最短にするには、どの製品を、どの製造装置で、どういう順番で製造するのが最適か

<pre>
品種 製品数 処理時間
(時間/ロット)
A 30 20時間
B 10 30時間
C 40 10時間
D 20 10時間
</pre>

<pre>
段取り時間
A→A 0時間
B→A 20時間
C→A 10時間
D→A 20時間
</pre>

In [1]:
# --- 問題設定 ---
machines = 10
products = ['A', 'B', 'C', 'D']
product_counts = {'A': 30, 'B': 10, 'C': 40, 'D': 20}
process_times = {'A': 20, 'B': 30, 'C': 10, 'D': 10}
setup_times = {
    ('A', 'A'): 0, ('B', 'A'): 20, ('C', 'A'): 10, ('D', 'A'): 20,
    ('A', 'B'): 20, ('B', 'B'): 0, ('C', 'B'): 20, ('D', 'B'): 10,
    ('A', 'C'): 10, ('B', 'C'): 10, ('C', 'C'): 0, ('D', 'C'): 20,
    ('A', 'D'): 20, ('B', 'D'): 10, ('C', 'D'): 20, ('D', 'D'): 0
}

T = 80  # 時間スロット上限（仮に設定）

In [2]:
from pyqubo import Array, Constraint, Placeholder

# 装置iのタイムスロットjに製品kを割り当てるかどうか (0/1)
x = Array.create('x', (machines, T, len(products)), 'BINARY')

# 目的関数1: マシンの総労働時間をなるべく最小化したい
process_cost = sum(
    process_times[products[p]] * x[m][t][p]
    for m in range(machines)
    for t in range(T)
    for p in range(len(products))
)

# 目的関数2: 部品の品種を交換する際の段取り時間をなるべく最小化したい
setup_cost = sum(
    setup_times[(products[p1], products[p2])] * x[m][t][p1] * x[m][t+1][p2]
    for m in range(machines)
    for t in range(T - 1)
    for p1 in range(len(products))
    for p2 in range(len(products))
)

# 制約条件: 各装置は1つのタイムスロットに割り当てられる製品は1個 (1-hot)
one_hot_constraint = Constraint(
    sum(
        sum(x[m][t][p1] * x[m][t][p2]
            for p1 in range(len(products)) 
            for p2 in range(len(products)) if p1 != p2)
        for m in range(machines)
        for t in range(T)
    ),
    label="one_product_per_slot"
)

# 制約条件: 生産する製品数は、想定通りであること (N-hot)
product_total_constraint = Constraint(
    sum(
        (sum(x[m][t][p] for m in range(machines) for t in range(T)) - product_counts[products[p]]) ** 2
        for p in range(len(products))
    ),
    label="product_total"
)

# --- モデル構築 ---
H = Placeholder("process_cost")*process_cost + Placeholder("setup_cost")*setup_cost \
    + Placeholder("lambda1") * one_hot_constraint \
    + Placeholder("lambda2") * product_total_constraint

model = H.compile()
feed_dict = {"process_cost":1.0, "setup_cost":1.0, "lambda1": 1.0, "lambda2": 30.0}
qubo, offset = model.to_qubo(feed_dict=feed_dict)


In [3]:
from neal import SimulatedAnnealingSampler 

# アニーリング
sampler = SimulatedAnnealingSampler()
sampleset = sampler.sample_qubo(qubo, num_reads=100)

# 最良解の取得
best = sampleset.first.sample
decoded = model.decode_sample(best, vartype='BINARY', feed_dict=feed_dict)

# 満たされていない制約があれば表示
if decoded.constraints(only_broken=True):
    print("Broken constraints:", decoded.constraints(only_broken=True))
else:
    print("All constraints satisfied.")

# 結果の取得
solution = decoded.sample

All constraints satisfied.


In [4]:
from datetime import datetime, timedelta
import pandas as pd
import plotly.express as px

# 装置の開始時刻を決定　適宜変更あり
start_time = datetime(2025, 4, 10, 8, 00)

In [5]:
def get_result_df(solution, machines, T, products, process_times):
    data = []

    # 解 (decoded.sample) に基づいてスケジュールを抽出
    for m in range(machines):
        for t in range(T):
            for p in range(len(products)):
                val = solution[f'x[{m}][{t}][{p}]']
                if val == 1:
                    product = products[p]
                    duration = process_times[product]
                    data.append({
                        "Machine": f"Machine {m}",
                        "Start": start_time + timedelta(hours=t),
                        "Finish": start_time + timedelta(hours=t) + timedelta(hours=duration),
                        "Duration": duration,
                        "Product": product
                    })

    # データがない場合は警告
    if not data:
        print("⚠️ 有効なデータがありません。")
        return

    # DataFrameに変換して並べ替え
    df = pd.DataFrame(data)
    df.sort_values(by=["Machine", "Start"], inplace=True)
    
    return df

In [6]:
def gannt_chart(df):

    # Plotlyでガントチャートを描画
    fig = px.timeline(
        df,
        x_start="Start",
        x_end="Finish",
        y="Machine",
        color="Product",
        title="生産スケジュール（ガントチャート）",
        labels={"Product": "製品種"}
    )

    fig.update_yaxes(autorange="reversed")
    fig.update_layout(
        xaxis_title="時間スロット",
        yaxis_title="製造装置",
        height=600
    )

    # チャート表示
    fig.show()

In [7]:
df = get_result_df(decoded.sample, machines, T, products, process_times)
gannt_chart(df)

In [8]:
df

,Machine,Start,Finish,Duration,Product
0,Machine 0,2025-04-10 10:00:00,2025-04-10 20:00:00,10,C
1,Machine 0,2025-04-10 21:00:00,2025-04-11 07:00:00,10,D
2,Machine 0,2025-04-11 09:00:00,2025-04-11 19:00:00,10,C
3,Machine 0,2025-04-11 15:00:00,2025-04-12 11:00:00,20,A
4,Machine 0,2025-04-11 20:00:00,2025-04-12 06:00:00,10,C
...,...,...,...,...,...
95,Machine 9,2025-04-12 17:00:00,2025-04-13 03:00:00,10,C
96,Machine 9,2025-04-13 00:00:00,2025-04-13 20:00:00,20,A
97,Machine 9,2025-04-13 07:00:00,2025-04-13 17:00:00,10,D
98,Machine 9,2025-04-13 09:00:00,2025-04-14 05:00:00,20,A


In [9]:
product_counts = df['Product'].value_counts()
product_counts

Product
C    40
A    30
D    20
B    10
Name: count, dtype: int64

In [10]:
# 各マシンごとにデータを並べ替え（先頭と末尾の時間を取得するため）
df_sorted = df.sort_values(by=['Machine', 'Start'])

# 各マシンごとに先頭のStartと末尾のFinishを取得
machine_time = df_sorted.groupby('Machine').agg(
    Start=('Start', 'first'),  # 先頭のStart
    Finish=('Finish', 'last')  # 末尾のFinish
).reset_index()

# 各マシンの総作業時間を計算（末尾のFinish - 先頭のStart）
machine_time['Total_Work_Time'] = (machine_time['Finish'] - machine_time['Start']).dt.total_seconds() / 3600  # 時間単位

# 結果を表示
print(machine_time)

     Machine               Start              Finish  Total_Work_Time
0  Machine 0 2025-04-10 10:00:00 2025-04-13 17:00:00             79.0
1  Machine 1 2025-04-10 09:00:00 2025-04-14 01:00:00             88.0
2  Machine 2 2025-04-10 11:00:00 2025-04-13 19:00:00             80.0
3  Machine 3 2025-04-10 16:00:00 2025-04-13 19:00:00             75.0
4  Machine 4 2025-04-11 12:00:00 2025-04-13 22:00:00             58.0
5  Machine 5 2025-04-10 10:00:00 2025-04-13 23:00:00             85.0
6  Machine 6 2025-04-10 09:00:00 2025-04-14 01:00:00             88.0
7  Machine 7 2025-04-10 14:00:00 2025-04-14 01:00:00             83.0
8  Machine 8 2025-04-10 10:00:00 2025-04-13 16:00:00             78.0
9  Machine 9 2025-04-10 18:00:00 2025-04-13 22:00:00             76.0


In [11]:
# 全体の作業時間を計算（最初の作業開始時刻と最後の作業終了時刻の差を計算）
overall_start = df['Start'].min()  # 最初のStart時刻
overall_finish = df['Finish'].max()  # 最後のFinish時刻

# 総作業時間を計算
total_work_time = (overall_finish - overall_start).total_seconds() / 3600  # 時間単位に変換

# 結果を表示
print(f"{overall_start} ~ {overall_finish}")
print(f"マシン全体の総作業時間: {total_work_time} 時間")

2025-04-10 09:00:00 ~ 2025-04-14 13:00:00
マシン全体の総作業時間: 100.0 時間
